# Train our open pitch keypoint model (RF-DETR Keypoint, Apache 2.0)

Replaces the Roboflow-hosted keypoint model (`football-field-detection/10`) with our own model.

1. Downloads dataset **version 11** of `football-field-detection` (once; kept in Drive afterwards).
2. Splits it **by match** once your frames of matches 11 + 12 are labelled (11 + 12 -> validation,
   HILAL-HAZM -> test, everything else -> train); until then it keeps Roboflow's split (old frames
   only, same data as the current Roboflow model: a like-for-like comparison). Points are fixed to
   pitch labels 1-32.
3. Trains **RF-DETR Keypoint** at 768 px on the T4 (672 or 576 px if 768 does not fit in memory) (about 2-5 h; early stopping), with left/right
   mirroring that swaps the mirrored pitch points. Checkpoints are copied to Drive every 10 minutes;
   after a disconnect, **Run all** again and training resumes.
4. Scores the model on your labelled frames of matches 11 + 12 (never trained on) and HILAL-HAZM
   (pixel error per point) and runs it on both CVAT clips, saving the keypoints for Claude to compare with the old model through the full pipeline.

**Runtime:** T4 GPU. After an out-of-memory error: **Runtime -> Restart session**, then Run all. Colab Secret `ROBOFLOW_API_KEY` is needed only the first time (to download the
dataset). **Runtime -> Run all**; do not edit cells. Results: `MyDrive/Playbook/field_model/v11/`.
Using another Google account? Share `MyDrive/Playbook` with it and add a shortcut to `Playbook` in its My Drive.

In [ ]:
# Cell 1 — GPU check, Drive, settings
import os, json, time, glob, shutil, subprocess, sys
g = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True) if shutil.which('nvidia-smi') else None
if g is None or g.returncode != 0:
    raise SystemExit('No GPU. Runtime -> Change runtime type -> T4 GPU, then re-run.')
print('GPU:', g.stdout.strip())
from google.colab import drive
drive.mount('/content/drive')
if not os.path.isdir('/content/drive/MyDrive/Playbook'):
    raise SystemExit('MyDrive/Playbook not found. Other Google account? Add a shortcut to the shared Playbook folder in My Drive.')
DEST = '/content/drive/MyDrive/Playbook/field_model/v11'  # one folder per dataset version
OUT = f'{DEST}/train'
os.makedirs(OUT, exist_ok=True)
WORKSPACE, PROJECT, VERSION = 'muwafag-hussain-o1ghr', 'football-field-detection-f07vi-it2xv', 11
# (resolution, batch, gradient accumulation): tried in order until one fits in the T4's memory
SETTINGS, EPOCHS = [(768, 1, 16), (672, 1, 16), (576, 1, 16)], 80
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # less memory fragmentation
FIELD_CONF, KP_CONF = 0.30, 0.50  # the pipeline's defaults
CLIPS = {'hilal_hazm_B': 'HILAL-HAZM_match_B_up10.mp4', 'hilal_ahli_B': 'HILAL-AHLI_match_B-up8.mp4'}

In [ ]:
# Cell 2 — Our repo + the two CVAT clips, and RF-DETR
PB, BRANCH = '/content/playbook', 'claude/setup-gpu-video-testing-JhgUH'
if not os.path.isdir(PB + '/.git'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', BRANCH,
                    'https://github.com/MuwafagQ/Playbook-program.git', PB], check=True)
else:  # a session that already ran: always use the latest code
    subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', BRANCH], cwd=PB, check=True)
    subprocess.run(['git', 'reset', '-q', '--hard', 'FETCH_HEAD'], cwd=PB, check=True)
for m in [m for m in list(sys.modules) if m.split('.')[0] in ('vision', 'tools', 'core', 'geometry')]:
    del sys.modules[m]  # re-import our code after an update
os.chdir(PB); sys.path.insert(0, PB)
VID = '/content/videos'; os.makedirs(VID, exist_ok=True)
subprocess.run(['git', 'fetch', '-q', '--depth', '1', 'origin', 'final-deliverable'], cwd=PB, check=True)
for f in CLIPS.values():
    if not os.path.exists(f'{VID}/{f}'):
        open(f'{VID}/{f}', 'wb').write(subprocess.run(['git', 'show', f'FETCH_HEAD:{f}'], cwd=PB,
                                                      capture_output=True, check=True).stdout)
!pip install -q "rfdetr[train]==1.11.0" roboflow
print('ready')

In [ ]:
# Cell 3 — Dataset version 11 (downloaded once, kept in Drive) -> split by match
SRC_DRIVE, SRC, SPLIT = f'{DEST}/v{VERSION}_coco', f'/content/field_v{VERSION}', '/content/field_split'
if not os.path.exists(f'{SRC_DRIVE}/train/_annotations.coco.json'):
    from google.colab import userdata
    from roboflow import Roboflow
    Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY')).workspace(WORKSPACE).project(PROJECT) \
        .version(VERSION).download('coco', location=SRC, overwrite=True)
    shutil.copytree(SRC, SRC_DRIVE, dirs_exist_ok=True)
elif not os.path.exists(f'{SRC}/train/_annotations.coco.json'):
    shutil.copytree(SRC_DRIVE, SRC, dirs_exist_ok=True)
from tools.keypoint_dataset import keypoint_dataset, FLIP_PAIRS
shutil.rmtree(SPLIT, ignore_errors=True)
stats = keypoint_dataset(SRC, SPLIT)
json.dump(stats, open(f'{DEST}/dataset_stats.json', 'w'), indent=1)
print(json.dumps(stats, indent=1))
if stats['valid']['images'] == 0:
    raise SystemExit('Validation split is empty: tell Claude (dataset version or file names changed).')
print('split by', stats['split_by'], '(match = your labelled frames of matches 11/12 are in; roboflow = old frames only)')

In [ ]:
# Cell 4 — Train RF-DETR Keypoint (resumes from the Drive checkpoint; copies to Drive every 10 min)
import threading, re
from rfdetr import RFDETRKeypointPreview
LOCAL = '/content/train_out'
os.makedirs(LOCAL, exist_ok=True)
FINISHED = os.path.exists(f'{OUT}/checkpoint_best_total.pth')  # written only when training ends
if not FINISHED and os.path.exists(f'{OUT}/last.ckpt') and not os.path.exists(f'{LOCAL}/last.ckpt'):
    shutil.copy(f'{OUT}/last.ckpt', f'{LOCAL}/last.ckpt')
    print('Resuming from the checkpoint in Drive.')

def sync_to_drive():
    for p in glob.glob(f'{LOCAL}/**/*', recursive=True):
        if not os.path.isfile(p) or re.fullmatch(r'checkpoint_\d+\.ckpt', os.path.basename(p)):
            continue  # periodic 500 MB backups: last.ckpt is enough to resume
        dst = os.path.join(OUT, os.path.relpath(p, LOCAL))
        if os.path.exists(dst) and os.path.getmtime(dst) >= os.path.getmtime(p):
            continue
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        shutil.copy(p, dst + '.tmp'); os.replace(dst + '.tmp', dst)

stop = threading.Event()
def loop():
    while not stop.wait(600):
        try:
            sync_to_drive()
        except Exception as e:
            print('Drive sync skipped:', e)
threading.Thread(target=loop, daemon=True).start()

import gc, torch
CHOSEN = f'{OUT}/train_setting.json'  # the setting that fits; a resumed run keeps it
if FINISHED:
    stop.set()
    print('Training already finished (model in Drive); skipping. Delete', OUT, 'to train again.')
else:
    t0 = time.time()
    options = [tuple(json.load(open(CHOSEN)))] if os.path.exists(CHOSEN) else SETTINGS
    try:
        for res, batch, accum in options:
            print(f'Training at {res} px, batch {batch} x {accum} steps')
            json.dump([res, batch, accum], open(CHOSEN, 'w'))
            model = None
            try:
                model = RFDETRKeypointPreview()
                model.train(dataset_dir=SPLIT, epochs=EPOCHS, batch_size=batch, grad_accum_steps=accum, lr=1e-4,
                            resolution=res, output_dir=LOCAL, early_stopping=True, checkpoint_interval=5,
                            keypoint_flip_pairs=FLIP_PAIRS,
                            resume=f'{LOCAL}/last.ckpt' if os.path.exists(f'{LOCAL}/last.ckpt') else None)
                break
            except RuntimeError as e:  # CUDA out of memory: try the next, smaller setting
                if 'out of memory' not in str(e).lower() or os.path.exists(f'{LOCAL}/last.ckpt') or res == options[-1][0]:
                    raise
                print(f'Out of GPU memory at {res} px; trying a smaller size.')
                os.remove(CHOSEN)
                del model; gc.collect(); torch.cuda.empty_cache()
                shutil.rmtree(LOCAL, ignore_errors=True); os.makedirs(LOCAL, exist_ok=True)
        print(f'Training done in {(time.time() - t0) / 3600:.1f} h')
    finally:
        stop.set()
        sync_to_drive()
        print('Checkpoints copied to Drive.')
print(sorted(os.path.basename(p) for p in glob.glob(f'{OUT}/*')))

In [ ]:
# Cell 5 — Score: pixel error on our labelled frames, keypoints on both CVAT clips, speed
import cv2, numpy as np, pandas as pd, torch
from vision.field_model import load_field_model
from vision.detect import infer_field_keypoints
from vision.preprocess import enhance_frame
best = next(p for p in [f'{OUT}/checkpoint_best_total.pth', f'{OUT}/checkpoint_best_ema.pth',
                        f'{OUT}/checkpoint_best_regular.pth'] if os.path.exists(p))
print('weights:', best)
field = load_field_model(best)

def keypoints(img):
    kp = infer_field_keypoints(field, enhance_frame(img, enabled=True), FIELD_CONF)
    return kp.class_id[0].tolist(), kp.xy[0].tolist(), kp.confidence[0].tolist()

# (a) labelled validation/test frames: error per point. *_ours = frames of our matches (valid = matches
# 11 + 12, never trained on: the main score; test = HILAL-HAZM, a match the old frames include);
# *_old = old frames (random split, near-duplicates of training frames: optimistic)
rows = []
for split in ('valid', 'test'):
    data = json.load(open(f'{SPLIT}/{split}/_annotations.coco.json'))
    for im in data['images']:
        img = cv2.imread(f"{SPLIT}/{split}/{im['file_name']}")
        labs, xys, cfs = keypoints(img)
        pred = {l: (xy, c) for l, xy, c in zip(labs, xys, cfs)}
        for a in [a for a in data['annotations'] if a['image_id'] == im['id']]:
            k = a['keypoints']
            for lab in range(1, 33):
                x, y, v = k[3 * (lab - 1):3 * lab]
                if v <= 0:
                    continue
                (px, py), c = pred.get(lab, ((np.nan, np.nan), 0.0))
                rows.append({'split': split + ('_ours' if im['file_name'].startswith(('match_video_', 'hilal_hazm'))
                                               else '_old'), 'image': im['file_name'], 'label': lab, 'gt_x': x, 'gt_y': y,
                             'x': px, 'y': py, 'conf': c, 'err_px': float(np.hypot(px - x, py - y))})
t = pd.DataFrame(rows); t.to_csv(f'{DEST}/scored_points.csv', index=False)
summary = {}
for split, g in t.groupby('split'):
    found = g[g.conf >= KP_CONF]
    summary[split] = {'frames': int(g.image.nunique()), 'points': len(g),
                      'found_share': round(len(found) / max(len(g), 1), 3),
                      'median_err_px': round(float(found.err_px.median()), 1) if len(found) else None,
                      'p90_err_px': round(float(found.err_px.quantile(0.9)), 1) if len(found) else None}
print(json.dumps(summary, indent=1))

# (b) both CVAT clips: every frame, as the pipeline runs it; saved for the full-pipeline comparison
speed = {}
for name, f in CLIPS.items():
    cap = cv2.VideoCapture(f'{VID}/{f}'); out, i, times = [], 0, []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        torch.cuda.synchronize(); t0 = time.time()
        labs, xys, cfs = keypoints(frame)
        torch.cuda.synchronize(); times.append(time.time() - t0)
        out += [{'frame': i, 'label': int(l), 'x': round(x, 1), 'y': round(y, 1), 'conf': round(c, 3)}
                for l, (x, y), c in zip(labs, xys, cfs)]
        i += 1
    cap.release()
    pd.DataFrame(out).to_csv(f'{DEST}/{name}_kp.csv', index=False)
    speed[name] = 1000 * float(np.median(times[5:]))
    print(f'{name}: {i} frames, {speed[name]:.1f} ms/frame')
summary.update({'ms_per_frame_median_T4': speed, 'train_setting': json.load(open(f'{OUT}/train_setting.json')) if os.path.exists(f'{OUT}/train_setting.json') else None, 'weights': os.path.basename(best)})
json.dump(summary, open(f'{DEST}/field_eval.json', 'w'), indent=1)
print(f'\nDone. Tell Claude the results are in {DEST}.')